# Module 00: Environment Setup and Orientation

Welcome to the fMRI analysis tutorial series. Before diving into neuroimaging data, we need to make sure
your Python environment has all the libraries you will need.

## Learning Objectives

By the end of this notebook you will be able to:

1. Verify that Python and conda are correctly configured.
2. Check whether all required and optional packages are importable.
3. Create and inspect a synthetic NIfTI image with **nibabel**.
4. Load and visualise a standard-space brain mask with **nilearn**.

---

> **Tip:** Run each cell with `Shift + Enter`. If a cell raises an `ImportError`, follow the
> installation instructions in Section 2 before continuing.

In [ ]:
import sys
import os
import platform

print("Python version  :", sys.version)
print("Platform        :", platform.platform())
print("Executable      :", sys.executable)
print("Conda env       :", os.environ.get("CONDA_DEFAULT_ENV", "(not a conda environment)"))

## 1. Checking Your Python Installation

The tutorial requires **Python ≥ 3.8**. The cell below confirms the version and whether you are
running inside a conda environment (recommended).

In [ ]:
import sys

major, minor, *_ = sys.version_info
if (major, minor) >= (3, 8):
    print(f"✅  Python {major}.{minor} — requirement satisfied.")
else:
    print(f"❌  Python {major}.{minor} — please upgrade to Python 3.8 or newer.")

conda_env = __import__('os').environ.get("CONDA_DEFAULT_ENV")
if conda_env:
    print(f"✅  Running inside conda environment: {conda_env}")
else:
    print("⚠️   No conda environment detected. Consider using conda for reproducibility.")

## 2. Installing Required Packages

If any packages are missing, install them with the commands below.

### Create the tutorial environment (recommended)

```bash
conda env create -f environments/environment_full.yml
conda activate fmri-tutorial-full
```

### Install individual packages

```bash
# Required
conda install -c conda-forge numpy pandas matplotlib nibabel

# Optional but strongly recommended
conda install -c conda-forge nilearn pybids nipype pydicom
pip install heudiconv
```

After installing, **restart the kernel** (`Kernel → Restart Kernel`) and re-run this notebook.

In [ ]:
packages = {
    "required": ["numpy", "pandas", "nibabel", "matplotlib"],
    "optional": ["nilearn", "bids", "nipype", "pydicom", "heudiconv"],
}

results = {}
for category, pkg_list in packages.items():
    print(f"\n--- {category.upper()} ---")
    for pkg in pkg_list:
        try:
            mod = __import__(pkg)
            version = getattr(mod, "__version__", "unknown")
            print(f"  ✅  {pkg:<15} version: {version}")
            results[pkg] = True
        except ImportError:
            print(f"  ❌  {pkg:<15} NOT FOUND")
            results[pkg] = False

missing_required = [p for p in packages["required"] if not results.get(p)]
if missing_required:
    print(f"\n⚠️   Missing required packages: {missing_required}")
    print("    Follow the installation instructions above before continuing.")
else:
    print("\n✅  All required packages are available!")

## 3. Testing nibabel

[nibabel](https://nipy.org/nibabel/) is the primary library for reading and writing neuroimaging
file formats (NIfTI, CIFTI, MINC, …). Let's create a small synthetic NIfTI image to confirm it
works correctly.

In [ ]:
import tempfile
import os
import numpy as np
import nibabel as nib

# Create a 10×10×10 array of random data
data = np.random.randn(10, 10, 10).astype(np.float32)

# Build a basic affine (1 mm isotropic, origin at centre)
affine = np.diag([1.0, 1.0, 1.0, 1.0])

img = nib.Nifti1Image(data, affine)
img.header.set_zooms((1.0, 1.0, 1.0))

# Save and reload from a temporary file
with tempfile.NamedTemporaryFile(suffix=".nii.gz", delete=False) as f:
    tmp_path = f.name

nib.save(img, tmp_path)
reloaded = nib.load(tmp_path)

print("Image shape    :", reloaded.shape)
print("Voxel sizes    :", reloaded.header.get_zooms())
print("Data dtype     :", reloaded.get_data_dtype())
print("Affine matrix  :")
print(reloaded.affine)
print("\n✅  nibabel is working correctly.")

os.unlink(tmp_path)

## 4. Testing nilearn

[nilearn](https://nilearn.github.io/) provides high-level tools for statistical learning on
neuroimaging data, along with convenient plotting utilities. We will load the MNI152 brain mask
(bundled with nilearn) and display it.

In [ ]:
try:
    import matplotlib
    matplotlib.use("Agg")  # non-interactive backend; change to 'inline' in JupyterLab
    import matplotlib.pyplot as plt
    from nilearn import datasets, plotting

    mask_img = datasets.load_mni152_brain_mask()
    print("Brain mask shape:", mask_img.shape)
    print("Voxel sizes     :", mask_img.header.get_zooms())

    display = plotting.plot_roi(
        mask_img,
        title="MNI152 Brain Mask (nilearn)",
        display_mode="ortho",
        cmap="cool",
    )
    plt.savefig("/tmp/mni_brain_mask.png", dpi=80, bbox_inches="tight")
    plt.close()
    print("\n✅  nilearn is working correctly. Figure saved to /tmp/mni_brain_mask.png")

except ImportError:
    print("⚠️   nilearn is not installed — skipping this test.")
    print("    Install it with:  conda install -c conda-forge nilearn")

## 5. Setting Up JupyterLab Extensions

Several JupyterLab extensions improve the neuroimaging analysis experience. Run the cell below
to see recommended setup commands.

In [ ]:
instructions = """
Recommended JupyterLab extensions
==================================

1. Variable inspector
   pip install lckr-jupyterlab-variableinspector

2. Table of Contents (built-in since JupyterLab 3)
   Available in the left sidebar under the list icon.

3. Niivue (interactive brain viewer)
   pip install niivue-jupyter
   jupyter labextension install niivue-jupyter  # if using JupyterLab < 3

4. Enable matplotlib inline rendering
   Add the following to the top of any notebook:
   %matplotlib inline

After installing extensions, restart JupyterLab for them to take effect.
"""
print(instructions)

## Summary and Next Steps

In this notebook you have:

- ✅ Confirmed Python ≥ 3.8 and a conda environment are active.
- ✅ Verified required packages (`numpy`, `pandas`, `nibabel`, `matplotlib`) are installed.
- ✅ Created, saved, and reloaded a synthetic NIfTI image with nibabel.
- ✅ Loaded and visualised the MNI152 brain mask with nilearn.

### Next Steps

Proceed to **Module 01: Understanding fMRI Data and the BIDS Standard**:

```
module_01_fmri_data_and_bids/
├── 01_understanding_fmri_data.ipynb   ← start here
└── 01_bids_overview.ipynb
```

You can also run the standalone verification script at any time:

```bash
python module_00_environment_setup/verify_installation.py
```